# Cross-attention for a padded batch

The lesson's NumPy block runs one head of cross-attention for a single pair. Training
runs on batches, where the inputs have different lengths and the shorter ones are padded.
Here you write cross-attention for a whole batch at once, with the input's pads hidden.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It makes a batch of three examples with 7, 4 and 6 real input tokens,
padded to 7, and 5 decoder positions each.

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
B, T_in, T_out, d_model, d_k = 3, 7, 5, 12, 3
enc_lengths = np.array([7, 4, 6])   # real input tokens in each example
H_enc = rng.standard_normal((B, T_in, d_model))    # encoder output
H_dec = rng.standard_normal((B, T_out, d_model))   # decoder states
scale = 1 / np.sqrt(d_model)
W_Q, W_K, W_V = scale * rng.standard_normal((3, d_model, d_k))

# The encoder writes a vector at every position, pads included.
# Here the pad vectors are large, so any weight on them shows.
for b in range(B):
    H_enc[b, enc_lengths[b]:] = 25.0
print("H_enc", H_enc.shape, "H_dec", H_dec.shape, "input lengths", enc_lengths)

## Part 1 (coding task): Masked attention for a batch

Write `masked_attention(Q, K, V, keep)`. `Q` has shape `(B, Tq, d)`, `K` has shape
`(B, Tk, d)` and `V` has shape `(B, Tk, dv)`. `keep` is a boolean array that broadcasts
to `(B, Tq, Tk)`, True where a query may attend to a key. Return the output, shape
`(B, Tq, dv)`, and the weights, shape `(B, Tq, Tk)`.

- Divide the scores by $\sqrt{d}$, as in the lesson.
- A hidden entry gets a weight of exactly 0.
- The weights of each query add up to 1.
- Large scores must not overflow.

You may assume that every query has at least one key it may attend to.

In [ ]:
def masked_attention(Q, K, V, keep):
    """Scaled dot-product attention for a batch. keep is True where a query may attend."""
    # TODO
    raise NotImplementedError

In [ ]:
ln = np.log
# One example small enough to check by hand: two queries, four keys, the last key hidden.
Q1 = np.array([[[2.0, 0, 0, 0], [0, 2.0, 0, 0]]])
K1 = np.array([[[0, ln(2), 0, 0], [ln(3), 0, 0, 0], [ln(4), 0, 0, 0], [90.0, 90.0, 0, 0]]])
V1 = np.array([[[1.0, 0, 0], [0, 1.0, 0], [0, 0, 1.0], [9.0, 9.0, 9.0]]])
out1, w1 = masked_attention(Q1, K1, V1, np.array([[[True, True, True, False]]]))
assert w1.shape == (1, 2, 4), f"the weights should have shape (1, 2, 4), got {w1.shape}"
assert out1.shape == (1, 2, 3), f"the output should have shape (1, 2, 3), got {out1.shape}"
assert np.allclose(w1.sum(axis=-1), 1), "the weights of each query should add up to 1 over the keys"
assert (w1[..., 3] == 0).all(), "the hidden key still gets some weight"
unscaled = [[[1 / 26, 9 / 26, 16 / 26, 0], [2 / 3, 1 / 6, 1 / 6, 0]]]
assert not np.allclose(w1, unscaled), "the scores were not divided by sqrt(d)"
expected = np.array([[[1 / 8, 3 / 8, 1 / 2, 0], [1 / 2, 1 / 4, 1 / 4, 0]]])
assert np.allclose(w1, expected), f"the weights should be {expected[0].tolist()}"
assert np.allclose(out1, expected[..., :3]), "the output should be the weighted sum of the value rows"

# A batch of two, with 3 queries and 5 keys. Example 1 hides its last two keys.
rng_b = np.random.default_rng(3)
Qb = rng_b.standard_normal((2, 3, 4))
Kb = rng_b.standard_normal((2, 5, 4))
Vb = rng_b.standard_normal((2, 5, 2))
keep_b = np.ones((2, 1, 5), dtype=bool)
keep_b[1, 0, 3:] = False
out_b, w_b = masked_attention(Qb, Kb, Vb, keep_b)
assert w_b.shape == (2, 3, 5) and out_b.shape == (2, 3, 2), f"got shapes {out_b.shape} and {w_b.shape} for 2 examples, 3 queries and 5 keys"
assert np.allclose(w_b.sum(axis=-1), 1), "with 3 queries and 5 keys, the weights of each query should add up to 1 over the 5 keys"
assert (w_b[1, :, 3:] == 0).all(), "a keep of shape (B, 1, Tk) should hide those keys from every query"
alone, _ = masked_attention(Qb[1:], Kb[1:, :3], Vb[1:, :3], np.ones((1, 1, 3), dtype=bool))
assert np.allclose(alone, out_b[1:]), "hiding two keys should give the same output as leaving them out"
big, w_big = masked_attention(100 * Qb, 100 * Kb, Vb, keep_b)
assert np.isfinite(big).all() and np.isfinite(w_big).all(), "large scores gave inf or NaN: hide the masked scores first, then subtract the largest remaining score in each row."
print("Part 1 passed")

## Part 2 (coding task): Cross-attention with padded inputs

Write `cross_attention(H_dec, H_enc, enc_lengths, W_Q, W_K, W_V)`, one head of
cross-attention for the whole batch. `H_dec` holds the decoder states, shape
`(B, T_out, d_model)`, and `H_enc` the encoder output, shape `(B, T_in, d_model)`.
Example $b$ has `enc_lengths[b]` real input tokens, followed by pads. Use your
`masked_attention`, and return its output and weights.

In [ ]:
def cross_attention(H_dec, H_enc, enc_lengths, W_Q, W_K, W_V):
    """One head of cross-attention for a batch whose inputs are padded."""
    # TODO
    raise NotImplementedError

In [ ]:
out, w = cross_attention(H_dec, H_enc, enc_lengths, W_Q, W_K, W_V)
assert out.shape == (B, T_out, d_k), f"the output should have one row per decoder position, shape {(B, T_out, d_k)}, got {out.shape}"
assert w.shape == (B, T_out, T_in), f"the weights should have shape {(B, T_out, T_in)}, got {w.shape}"
for b in range(B):
    n = enc_lengths[b]
    assert (w[b, :, n:] == 0).all(), f"example {b} puts weight on its pads"
    assert (w[b, :, :n] > 0).all(), f"example {b} hides some of its real input tokens"
H_other = H_enc.copy()
for b in range(B):
    H_other[b, enc_lengths[b]:] = -40.0
out_other, _ = cross_attention(H_dec, H_other, enc_lengths, W_Q, W_K, W_V)
assert np.allclose(out, out_other), "changing the pad vectors changed the output"
for b in range(B):
    n = enc_lengths[b]
    solo, _ = cross_attention(H_dec[b : b + 1], H_enc[b : b + 1, :n], np.array([n]), W_Q, W_K, W_V)
    assert np.allclose(solo[0], out[b]), f"example {b} comes out differently on its own, without pads"
assert np.allclose(out[1, 2], [-0.11473, 0.516638, -0.781491], atol=1e-5), "the outputs are wrong: check which states each of W_Q, W_K and W_V multiplies"
print("Part 2 passed")

## Solution

### Part 1: Masked attention for a batch

In [ ]:
def masked_attention(Q, K, V, keep):
    """Scaled dot-product attention for a batch. keep is True where a query may attend."""
    grid = np.einsum("bqd,bkd->bqk", Q, K) / np.sqrt(Q.shape[-1])   # (B, Tq, Tk)
    grid = np.where(keep, grid, -np.inf)              # a hidden entry gets e^(-inf) = 0
    raw = np.exp(grid - grid.max(axis=-1, keepdims=True))
    w = raw / raw.sum(axis=-1, keepdims=True)         # each query's row adds up to 1
    return w @ V, w

**Why it works.** `np.einsum("bqd,bkd->bqk", Q, K)` takes the dot product of every query
with every key of the same example, giving scores of shape `(B, Tq, Tk)`. The same grid
comes from `Q @ np.swapaxes(K, -1, -2)`, but not from `Q @ K.T`: `.T` reverses all three
axes, batch included, and multiplies the wrong things. A hidden score set to $-\infty$ gives $e^{-\infty} = 0$, so it gets weight
exactly 0 and the other weights still add up to 1. Zeroing weights after the softmax
would leave those rows short of 1, and multiplying the scores by `keep` would give the
hidden keys a score of 0, not a weight of 0. The softmax runs along the last axis, the
keys. In the lesson's single grid that axis is `axis=1`, but in a batch `axis=1` is the
queries. Subtracting each row's largest score keeps every exponent at or below 0.

### Part 2: Cross-attention with padded inputs

In [ ]:
def cross_attention(H_dec, H_enc, enc_lengths, W_Q, W_K, W_V):
    """One head of cross-attention for a batch whose inputs are padded."""
    T_in = H_enc.shape[1]
    real = np.arange(T_in) < np.asarray(enc_lengths)[:, None]   # (B, T_in)
    Q = H_dec @ W_Q                                              # queries from the decoder
    K, V = H_enc @ W_K, H_enc @ W_V                              # keys and values from the encoder
    return masked_attention(Q, K, V, real[:, None, :])           # same columns hidden in every row

**Why it works.** The queries come from the decoder states and the keys and values from
the encoder output, so the grid has one row per decoder position and one column per input
token, $5 \times 7$ here. `np.arange(T_in) < enc_lengths[:, None]` is True at each
example's real tokens, shape `(B, T_in)`, and the extra axis makes it `(B, 1, T_in)`:
the same columns hidden in every row. Nothing hides any decoder position, since the input
is complete before decoding starts. The pads get weight exactly 0, so whatever the encoder
wrote there never reaches the output, and each example comes out as it would alone.